# REINVENT4 + Case v2.1 — Pareto v6

Один Colab notebook для нового набора **7 evaluator-моделей** из `suharevalexey/Case` commit `6164ac0950bf903b8d67e2aaae8c6102af11176c`.

Что изменено относительно старой версии:
- 7 новых surrogate evaluator-ов вместо старых 6 моделей;
- точные признаки текущего Case: **1024-bit Morgan radius 2 + 12 RDKit descriptors = 1036**;
- текущие пороги Case и AD cutoff `0.70`;
- **7 независимых Pareto-профилей** — по одному приоритету на каждый evaluator;
- приоритет используется только внутри одинакового Pareto-front, поэтому не может перекрыть Pareto-доминирование по остальным свойствам;
- `ΔH` считается **по формуле Эйринга из прогнозируемого `t1/2`** как `ΔH‡`-proxy при `ΔS‡≈0`. Это не термодинамическая `ΔH_storage` между фотоизомерами;
- oracle-модели не скачиваются и не используются в обучении;
- evaluator-модели загружаются один раз в persistent local daemon, чтобы не перечитывать большие RandomForest на каждом RL-шаге.

Перед запуском включите GPU: **Runtime → Change runtime type → GPU**, затем **Run all**.


In [ ]:
# ===================== НАСТРОЙКИ ЭКСПЕРИМЕНТА =====================
# Каждый из 7 Pareto-профилей стартует из одного и того же prior и с одним seed.
SEED = 42
STEPS_PER_PROFILE = 60      # Для максимально полного прогона можно поставить 100
BATCH_SIZE = 64
SAMPLE_PER_PROFILE = 500    # Для полного финального отбора можно поставить 1000
SMOKE_STEPS = 2
SMOKE_BATCH = 16
# ==================================================================


In [ ]:
# Встроенный кодовый комплект Pareto v6. Модели evaluator скачиваются из pinned Case commit.
import base64, hashlib, io, shutil, zipfile
from pathlib import Path

_BUNDLE_SHA256 = "49c9293a875f62c4446a8fcc11d9cc2017c28b7b7a45551ea4bdfe742a3f00de"
_BUNDLE_B64 = """..."""

raw = base64.b64decode(_BUNDLE_B64)
assert hashlib.sha256(raw).hexdigest() == _BUNDLE_SHA256
ROOT = Path("/content/REINVENT4_MOST_PARETO_V6")
if ROOT.exists():
    # Код обновляем, тяжёлые окружения/модели переиспользуются run.py, если совместимы.
    pass
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    z.extractall("/content")
print("Bundle SHA256 OK:", _BUNDLE_SHA256)
print("ROOT:", ROOT)


In [ ]:
import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path

ROOT = Path('/content/REINVENT4_MOST_PARETO_V6')
LOGDIR = ROOT / 'colab_logs'
LOGDIR.mkdir(parents=True, exist_ok=True)


def banner(text):
    print('\n' + '='*92)
    print(text)
    print('='*92, flush=True)


def run_live(args, log_name, cwd=ROOT):
    args=[str(x) for x in args]
    print('\n>>>', ' '.join(args), flush=True)
    log_path=LOGDIR/log_name
    print('Лог:', log_path, flush=True)
    with log_path.open('w', encoding='utf-8') as log:
        p=subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, encoding='utf-8', errors='replace', bufsize=1)
        tail=[]
        for line in p.stdout:
            print(line, end='')
            log.write(line); log.flush()
            tail.append(line.rstrip())
            if len(tail)>160:
                tail=tail[-160:]
        rc=p.wait()
    if rc:
        raise RuntimeError('Команда завершилась с кодом %d\n%s' % (rc, '\n'.join(tail[-120:])))
    return log_path


banner('1/6  GPU + Python 3.12')
GPU = shutil.which('nvidia-smi') is not None
PROCESSOR = 'cu126' if GPU else 'cpu'
DEVICE = 'cuda:0' if GPU else 'cpu'
print('GPU:', GPU, 'PROCESSOR:', PROCESSOR, 'DEVICE:', DEVICE)
if GPU:
    subprocess.run(['nvidia-smi'], check=False)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
UV=shutil.which('uv')
if not UV:
    raise RuntimeError('uv не найден')
subprocess.run([UV, 'python', 'install', '3.12'], check=True)
PY312=subprocess.run([UV, 'python', 'find', '3.12'], check=True, text=True, capture_output=True).stdout.strip()
print('Python 3.12:', PY312)

banner('2/6  Setup: REINVENT4 + 7 новых Case evaluator-ов')
run_live([PY312, 'run.py', 'setup', '--processor', PROCESSOR], 'setup.log')

banner('3/6  Полная проверка ExternalProcess / Pareto scorer')
run_live([PY312, 'run.py', 'check', '--seed', str(SEED)], 'check.log')

banner('4/6  Smoke-test RL')
run_live([
    PY312, 'run.py', 'smoke',
    '--steps', str(SMOKE_STEPS),
    '--batch-size', str(SMOKE_BATCH),
    '--device', DEVICE,
    '--seed', str(SEED),
], 'smoke.log')

banner('5/6  Семь независимых Pareto-профилей')
run_live([
    PY312, 'run.py', 'experiment',
    '--steps', str(STEPS_PER_PROFILE),
    '--batch-size', str(BATCH_SIZE),
    '--n', str(SAMPLE_PER_PROFILE),
    '--device', DEVICE,
    '--seed', str(SEED),
], 'experiment.log')

banner('6/6  Итоги')
latest=json.loads((ROOT/'runs/latest_pareto_experiment.json').read_text(encoding='utf-8'))
exp=Path(latest['experiment'])
summary=json.loads((exp/'aggregate/summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))

summary_csv=exp/'aggregate/profile_summary.csv'
try:
    import pandas as pd
    display(pd.read_csv(summary_csv))
    front=pd.read_csv(exp/'aggregate/global_pareto_front.csv')
    print('Global Pareto front rows:', len(front))
    display(front.head(30))
except Exception as exc:
    print('Табличный preview пропущен:', repr(exc))

RESULTS=Path('/content/REINVENT4_MOST_PARETO_RESULTS.zip')
CHECKPOINTS=Path('/content/REINVENT4_MOST_PARETO_CHECKPOINTS.zip')
assert RESULTS.is_file(), RESULTS
assert CHECKPOINTS.is_file(), CHECKPOINTS
print('\nГОТОВО')
print('Результаты:', RESULTS)
print('Checkpoints:', CHECKPOINTS)
print('Эксперимент:', exp)


## После завершения

Основной лёгкий архив с CSV/логами/Pareto-front:

`/content/REINVENT4_MOST_PARETO_RESULTS.zip`

Отдельный тяжёлый архив с 7 обученными агентами:

`/content/REINVENT4_MOST_PARETO_CHECKPOINTS.zip`

Ключевые таблицы внутри результатов:
- `profile_summary.csv` — сравнение 7 приоритетов;
- `global_pareto_front.csv` — глобальный Pareto-front после объединения всех профилей;
- `strict_candidates.csv` — кандидаты, прошедшие все 7 raw-порогов + обе AD + SAScore;
- `robust_candidates.csv` — более строгий вариант с учётом `0.5σ` неопределённости evaluator-ов.
